# 03 — Correlation analysis

## Pertanyaan

Indikator nasional dan moneter mana yang bergerak bersama secara statistik, dan bagaimana korelasinya berubah ketika indikator penjelas digeser beberapa tahun?

## Metode

Korelasi Pearson dihitung secara pairwise beserta jumlah observasi yang overlap. Data BI bulanan dirata-ratakan per tahun sebelum digabungkan. Untuk lagged correlation, lag positif berarti target pada tahun *t* dibandingkan dengan fitur dari tahun sebelumnya. Korelasi tidak ditafsirkan sebagai kausalitas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from analytics.descriptive.data_access import DATASET_SOURCES
from analytics.descriptive.notebook_support import insight, prepare_notebook, save_figure
from analytics.descriptive.statistics import build_correlation_dataset, correlation_with_overlap, lagged_correlation

data = prepare_notebook()
correlation_data = build_correlation_dataset(data.national, data.monetary)
correlation_columns = [
    "gdp_growth_percent",
    "inflation_percent",
    "unemployment_percent",
    "population",
    "gdp_per_capita_current_usd",
    "bi_rate_annual_average",
    "jisdor_annual_average",
]
correlations, overlap = correlation_with_overlap(correlation_data, correlation_columns)

## Hasil

In [ ]:
display(correlations)
print("Jumlah observasi pairwise:")
display(overlap)

fig, ax = plt.subplots(figsize=(10, 8))
matrix = correlations.to_numpy(dtype=float)
image = ax.imshow(np.ma.masked_invalid(matrix), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(correlation_columns)), correlation_columns, rotation=60, ha="right")
ax.set_yticks(range(len(correlation_columns)), correlation_columns)
for row in range(len(correlation_columns)):
    for column in range(len(correlation_columns)):
        if np.isfinite(matrix[row, column]):
            ax.text(column, row, f"{matrix[row, column]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Matriks korelasi Pearson")
fig.colorbar(image, ax=ax, label="Koefisien korelasi")
fig.tight_layout()
save_figure(fig, "03_correlation_matrix.png")
display(fig)
plt.close(fig)

In [ ]:
lags = range(-3, 4)
inflation_lags = lagged_correlation(
    correlation_data, "gdp_growth_percent", "inflation_percent", lags
)
bi_rate_lags = lagged_correlation(
    correlation_data, "gdp_growth_percent", "bi_rate_annual_average", lags
)
display(inflation_lags)
display(bi_rate_lags)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(inflation_lags["lag"], inflation_lags["correlation"], marker="o", label="Inflasi")
ax.plot(bi_rate_lags["lag"], bi_rate_lags["correlation"], marker="o", label="Rata-rata BI-Rate")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Lagged correlation terhadap pertumbuhan GDP", xlabel="Lag fitur (tahun)", ylabel="Korelasi Pearson", xticks=list(lags))
ax.legend()
fig.tight_layout()
save_figure(fig, "03_lagged_correlation.png")
display(fig)
plt.close(fig)

In [ ]:
upper_triangle = correlations.where(np.triu(np.ones(correlations.shape), k=1).astype(bool))
pairs = upper_triangle.stack().rename("correlation").reset_index()
strongest = pairs.loc[pairs["correlation"].abs().idxmax()]
pair_overlap = int(overlap.at[strongest["level_0"], strongest["level_1"]])
display(insight(
    f"Korelasi absolut pairwise terbesar adalah {strongest['correlation']:.3f} antara {strongest['level_0']} dan {strongest['level_1']}, berdasarkan {pair_overlap} observasi yang overlap.",
    frame=correlation_data,
    source=f"{DATASET_SOURCES['national']}; {DATASET_SOURCES['monetary']}",
    limitation="Korelasi tidak menunjukkan kausalitas. Tren bersama, sampel kecil, perbedaan frekuensi, dan variabel lain dapat menghasilkan hubungan semu.",
))
best_inflation_lag = inflation_lags.dropna().loc[inflation_lags.dropna()["correlation"].abs().idxmax()]
display(insight(
    f"Untuk inflasi dan pertumbuhan GDP, korelasi absolut terbesar pada rentang lag yang diuji adalah {best_inflation_lag['correlation']:.3f} pada lag {int(best_inflation_lag['lag'])}, dengan {int(best_inflation_lag['observation_count'])} pasangan observasi.",
    frame=data.national,
    source=DATASET_SOURCES["national"],
    limitation="Pemilihan lag setelah melihat beberapa kandidat meningkatkan risiko pola kebetulan dan tetap tidak memberi bukti kausal.",
))

## Interpretasi

Koefisien menunjukkan arah dan kekuatan hubungan linear pada observasi yang tersedia. Matriks overlap harus dibaca bersama koefisien karena pasangan yang melibatkan data BI memiliki periode lebih pendek.

## Keterbatasan

Tidak ada kontrol terhadap confounder, perubahan struktur ekonomi, nonstationarity, multiple testing, atau hubungan nonlinear. Lag adalah pergeseran observasi tahunan dan tidak membuktikan urutan sebab-akibat.